In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
from src.simulation import simulate_price, run_market_maker

def evaluate(distance=0.01, skew=0.0, informed_rate=0.0, horizon=60,
             n_hours=100, seed=1):
    rng = np.random.default_rng(seed)
    pnls, max_inv = [], []
    for _ in range(n_hours):
        prices = simulate_price(rng)
        result = run_market_maker(rng, prices, distance=distance, skew=skew,
                                  informed_rate=informed_rate, horizon=horizon)
        pnls.append(result["pnl"].iloc[-1])
        max_inv.append(result["inventory"].abs().max())
    pnls = np.array(pnls)
    return {
        "skew": skew, "distance": distance,
        "informed_rate": informed_rate, "horizon": horizon,
        "Avg PnL": pnls.mean(), "Std": pnls.std(),
        "Avg PnL / Std": pnls.mean() / pnls.std(),
        "Losing hours": (pnls < 0).mean(), "Worst hour": pnls.min(),
        "Avg max inventory": np.mean(max_inv),
    }

# Result 2: naive market maker, 1,000 hours
print("=== RESULT 2: naive market maker ===")
print(pd.Series(evaluate(n_hours=1000, seed=42)).round(2))

# Result 3: inventory skew, 300 hours per row
print("\n=== RESULT 3: inventory skew ===")
skews = [0, 0.0001, 0.0002, 0.0005, 0.001, 0.002, 0.005, 0.01, 0.02]
table3 = pd.DataFrame([evaluate(skew=s, n_hours=300) for s in skews])
print(table3[["skew", "Avg PnL", "Std", "Avg PnL / Std",
              "Losing hours", "Worst hour", "Avg max inventory"]].round(4))

# Result 4a: informed rate x distance, skew 0.002
print("\n=== RESULT 4a: adverse selection ===")
rows = [evaluate(distance=d, skew=0.002, informed_rate=r)
        for r in [0, 0.02, 0.05, 0.1] for d in [0.01, 0.02, 0.03, 0.05]]
table4a = pd.DataFrame(rows).pivot(index="informed_rate", columns="distance",
                                   values="Avg PnL")
print(table4a.round(2))

# Result 4b: skew x horizon
print("\n=== RESULT 4b: skew vs speed of information ===")
rows = [evaluate(skew=s, horizon=h, informed_rate=r)
        for s in [0, 0.002] for h in [5, 60] for r in [0, 0.1]]
table4b = pd.DataFrame(rows)
print(table4b[["skew", "horizon", "informed_rate",
               "Avg PnL", "Losing hours"]].round(3)) 

=== RESULT 2: naive market maker ===
skew                  0.00
distance              0.01
informed_rate         0.00
horizon              60.00
Avg PnL              14.41
Std                  14.54
Avg PnL / Std         0.99
Losing hours          0.12
Worst hour          -59.27
Avg max inventory    42.10
dtype: float64

=== RESULT 3: inventory skew ===
     skew  Avg PnL      Std  Avg PnL / Std  Losing hours  Worst hour  \
0  0.0000  13.7062  14.1199         0.9707        0.1267    -26.4230   
1  0.0001  13.5415   3.8225         3.5425        0.0000      2.7602   
2  0.0002  13.4785   2.6969         4.9978        0.0000      5.9532   
3  0.0005  13.2770   1.7366         7.6454        0.0000      8.5981   
4  0.0010  13.0390   1.2632        10.3221        0.0000      9.5427   
5  0.0020  12.7196   0.9184        13.8501        0.0000     10.5634   
6  0.0050  11.6611   0.6106        19.0962        0.0000      9.6172   
7  0.0100   9.3745   0.4804        19.5143        0.0000      8.0967

In [2]:
print(table3[["skew", "Avg max inventory"]].round(1))

   skew  Avg max inventory
0   0.0               41.3
1   0.0               16.8
2   0.0               13.0
3   0.0                9.0
4   0.0                6.7
5   0.0                5.0
6   0.0                3.1
7   0.0                2.0
8   0.0                1.0
